# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process a dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All references to record sets, fields, and columns are made by their unique `@id` identifiers as defined in the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:  
<https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json>

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
We load the dataset using the Croissant schema URL and inspect its basic metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load Croissant metadata
dataset = mlc.Dataset(croissant_url)

# Access and print basic metadata
metadata = dataset.metadata
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Explore available record sets and their fields using their `@id` fields, following the Croissant specification.

First, list all record sets in the dataset and display their fields by `@id`.

In [ ]:
# List all record sets and their field @id s
print("Available Record Sets (by @id):")
record_sets = []
for rs in dataset.metadata.record_sets:
    print(f"  RecordSet @id: {rs.id}")
    record_sets.append(rs.id)
    print("    Fields:")
    for field in rs.fields:
        print(f"      Field @id: {field.id} (name: {field.name}, data_type: {field.data_type})")
    print()
# For demonstration, let's choose the first record set
if len(record_sets) > 0:
    chosen_record_set_id = record_sets[0]
    print(f"Chosen record_set @id for extraction: {chosen_record_set_id}")
else:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load data from each record set into DataFrames for analysis. We use the `record_set` `@id` and reference fields by their unique `@id`. Replace with your selected record set(s) if desired.

In [ ]:
# Load data from all available record sets by their @id
all_dataframes = {}
for rs_id in record_sets:
    # Use generator to extract the records as dicts
    try:
        records = list(dataset.records(record_set=rs_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            all_dataframes[rs_id] = df
            print(f"Loaded DataFrame for RecordSet @id: {rs_id} with {df.shape[0]} rows and {df.shape[1]} columns.")
            print(df.head(2))
        else:
            print(f"RecordSet @id {rs_id} is empty.")
    except Exception as e:
        print(f"Could not load data for RecordSet @id: {rs_id}")
        print(e)

# Pick a main record set for further exploration (e.g., the first one that was loaded)
if len(all_dataframes) > 0:
    main_record_set_id = list(all_dataframes.keys())[0]
    df = all_dataframes[main_record_set_id]
    print(f"Main analysis will use RecordSet @id: {main_record_set_id}")
    print(f"Columns (@id): {df.columns.tolist()}")
    display(df.head())
else:
    main_record_set_id = None
    print("No DataFrames loaded. Cannot proceed.")

## 4. Exploratory Data Analysis (EDA)
Let's demonstrate EDA operations such as filtering, normalization, and grouping. This section uses unique field `@id`s extracted from the dataset.

**You should set `numeric_field_id` and `group_field_id` to valid column `@id`s from the previous data extraction.**

In [ ]:
if main_record_set_id is not None:
    df = all_dataframes[main_record_set_id]
    # Attempt to identify numeric columns by dtype
    numeric_columns = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print("Available numeric columns (by @id):", numeric_columns)
    # If at least one numeric column exists, analyze it
    if numeric_columns:
        # Use the first numeric field by @id
        numeric_field_id = numeric_columns[0]
        print(f"Using numeric field @id: {numeric_field_id} for analysis.")
        threshold = df[numeric_field_id].median()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered rows with {numeric_field_id} > {threshold}\n", filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a non-numeric column @id, if available
        candidate_group_fields = [col for col in df.columns if pd.api.types.is_object_dtype(df[col])]
        if candidate_group_fields:
            group_field_id = candidate_group_fields[0]
            print(f"Grouping by field @id: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No categorical string fields found for grouping.")
    else:
        print("No numeric columns found in this record set.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field and optionally a relationship with a group or category field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if main_record_set_id is not None and len(numeric_columns) > 0:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()
    # Pairplot if grouping field exists
    if 'group_field_id' in locals():
        plt.figure(figsize=(9, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"Boxplot of {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- Demonstrated loading and exploring a Croissant dataset with the `mlcroissant` library.
- Showed how to inspect record sets and fields using their `@id` identifiers.
- Loaded data into DataFrames, performed simple EDA and visualized key relationships.

For deeper analysis, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/).